# Projeto Airbnb Rio - Ferramenta de Previsão de Preço de Imóvel para pessoas comuns 

### Contexto

No Airbnb, qualquer pessoa que tenha um quarto ou um imóvel de qualquer tipo (apartamento, casa, chalé, pousada, etc.) pode ofertar o seu imóvel para ser alugado por diária.

Você cria o seu perfil de host (pessoa que disponibiliza um imóvel para aluguel por diária) e cria o anúncio do seu imóvel.

Nesse anúncio, o host deve descrever as características do imóvel da forma mais completa possível, de forma a ajudar os locadores/viajantes a escolherem o melhor imóvel para eles (e de forma a tornar o seu anúncio mais atrativo)

Existem dezenas de personalizações possíveis no seu anúncio, desde quantidade mínima de diária, preço, quantidade de quartos, até regras de cancelamento, taxa extra para hóspedes extras, exigência de verificação de identidade do locador, etc.

### Nosso objetivo

Construir um modelo de previsão de preço que permita uma pessoa comum que possui um imóvel possa saber quanto deve cobrar pela diária do seu imóvel.

Ou ainda, para o locador comum, dado o imóvel que ele está buscando, ajudar a saber se aquele imóvel está com preço atrativo (abaixo da média para imóveis com as mesmas características) ou não.

### O que temos disponível, inspirações e créditos

As bases de dados foram retiradas do site kaggle: https://www.kaggle.com/allanbruno/airbnb-rio-de-janeiro

Elas estão disponíveis para download abaixo da aula (se você puxar os dados direto do Kaggle pode ser que encontre resultados diferentes dos meus, afinal as bases de dados podem ter sido atualizadas).

Caso queira uma outra solução, podemos olhar como referência a solução do usuário Allan Bruno do kaggle no Notebook: https://www.kaggle.com/allanbruno/helping-regular-people-price-listings-on-airbnb

Você vai perceber semelhanças entre a solução que vamos desenvolver aqui e a dele, mas também algumas diferenças significativas no processo de construção do projeto.

- As bases de dados são os preços dos imóveis obtidos e suas respectivas características em cada mês.
- Os preços são dados em reais (R$)
- Temos bases de abril de 2018 a maio de 2020, com exceção de junho de 2018 que não possui base de dados

### Expectativas Iniciais

- Acredito que a sazonalidade pode ser um fator importante, visto que meses como dezembro costumam ser bem caros no RJ
- A localização do imóvel deve fazer muita diferença no preço, já que no Rio de Janeiro a localização pode mudar completamente as características do lugar (segurança, beleza natural, pontos turísticos)
- Adicionais/Comodidades podem ter um impacto significativo, visto que temos muitos prédios e casas antigos no Rio de Janeiro

Vamos descobrir o quanto esses fatores impactam e se temos outros fatores não tão intuitivos que são extremamente importantes.

In [21]:
#importando bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
import numpy
import os

#ler base de dados
lista_de_dados = os.listdir("dataset")

dataset = pd.DataFrame()

for dados in lista_de_dados:
    novo_df = pd.read_csv(f"dataset/{dados}", low_memory=False)
    dataset = pd.concat([dataset, novo_df])

dataset = dataset.reset_index(drop=True)

display(dataset)

,id,listing_url,scrape_id,last_scraped,name,summary,space,description,experiences_offered,neighborhood_overview,...,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,number_of_reviews_ltm,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms
0,14063,https://www.airbnb.com/rooms/14063,20180414160018,2018-04-14,Living in a Postcard,"Besides the most iconic's view, our apartment ...",NaN,"Besides the most iconic's view, our apartment ...",none,Best and favorite neighborhood of Rio. Perfect...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,17878,https://www.airbnb.com/rooms/17878,20180414160018,2018-04-14,Very Nice 2Br - Copacabana - WiFi,Please note that special rates apply for New Y...,- large balcony which looks out on pedestrian ...,Please note that special rates apply for New Y...,none,This is the best spot in Rio. Everything happe...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,24480,https://www.airbnb.com/rooms/24480,20180414160018,2018-04-14,Nice and cozy near Ipanema Beach,My studio is located in the best of Ipanema. ...,The studio is located at Vinicius de Moraes St...,My studio is located in the best of Ipanema. ...,none,"The beach, the lagoon, Ipanema is a great loca...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,25026,https://www.airbnb.com/rooms/25026,20180414160018,2018-04-14,Beautiful Modern Decorated Studio in Copa,"Our apartment is a little gem, everyone loves ...",This newly renovated studio (last renovations ...,"Our apartment is a little gem, everyone loves ...",none,Copacabana is a lively neighborhood and the ap...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,31560,https://www.airbnb.com/rooms/31560,20180414160018,2018-04-14,NICE & COZY 1BDR - IPANEMA BEACH,This nice and clean 1 bedroom apartment is loc...,This nice and clean 1 bedroom apartment is loc...,This nice and clean 1 bedroom apartment is loc...,none,Die Nachbarschaft von Ipanema ist super lebend...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
902205,38844730,https://www.airbnb.com/rooms/38844730,20190923212307,2019-09-24,TRANSAMERICA BARRA DA TIJUCA R IV,"Em estilo contemporâneo, o Transamerica Prime ...",NaN,"Em estilo contemporâneo, o Transamerica Prime ...",none,NaN,...,1.0,1.0,1125.0,1125.0,1.0,1125.0,0.0,15.0,0.0,0.0
902206,38846408,https://www.airbnb.com/rooms/38846408,20190923212307,2019-09-24,Alugo para o Rock in Rio,"Confortável apartamento, 2 quartos , sendo 1 s...",O apartamento estará com mobília completa disp...,"Confortável apartamento, 2 quartos , sendo 1 s...",none,"Muito próximo ao Parque Olímpico, local do eve...",...,2.0,2.0,1125.0,1125.0,2.0,1125.0,0.0,1.0,0.0,0.0
902207,38846703,https://www.airbnb.com/rooms/38846703,20190923212307,2019-09-24,Apt COMPLETO em COPACABANA c/TOTAL SEGURANÇA,Apartamento quarto e sala COMPLETO para curtas...,Espaço ideal para até 5 pessoas. Cama de casal...,Apartamento quarto e sala COMPLETO para curtas...,none,NaN,...,3.0,3.0,1125.0,1125.0,3.0,1125.0,0.0,23.0,6.0,0.0
902208,38847050,https://www.airbnb.com/rooms/38847050,20190923212307,2019-09-24,Cobertura Cinematografica,Cobertura alto nivel,NaN,Cobertura alto nivel,none,NaN,...,1.0,1.0,1125.0,1125.0,1.0,1125.0,0.0,1.0,0.0,0.0


In [22]:
#analisar base
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 902210 entries, 0 to 902209
Columns: 106 entries, id to calculated_host_listings_count_shared_rooms
dtypes: float64(31), int64(13), object(1), str(61)
memory usage: 729.6+ MB


In [23]:
for coluna in dataset.columns:
    print(coluna)

id
listing_url
scrape_id
last_scraped
name
summary
space
description
experiences_offered
neighborhood_overview
notes
transit
access
interaction
house_rules
thumbnail_url
medium_url
picture_url
xl_picture_url
host_id
host_url
host_name
host_since
host_location
host_about
host_response_time
host_response_rate
host_acceptance_rate
host_is_superhost
host_thumbnail_url
host_picture_url
host_neighbourhood
host_listings_count
host_total_listings_count
host_verifications
host_has_profile_pic
host_identity_verified
street
neighbourhood
neighbourhood_cleansed
neighbourhood_group_cleansed
city
state
zipcode
market
smart_location
country_code
country
latitude
longitude
is_location_exact
property_type
room_type
accommodates
bathrooms
bedrooms
beds
bed_type
amenities
square_feet
price
weekly_price
monthly_price
security_deposit
cleaning_fee
guests_included
extra_people
minimum_nights
maximum_nights
calendar_updated
has_availability
availability_30
availability_60
availability_90
availability_365
cal

In [24]:
#limpando a base de dados
colunas_remover = [
    "name",
    "summary",
    "space",
    "description",
    "neighborhood_overview",
    "notes",
    "transit",
    "access",
    "interaction",
    "house_rules",

    "listing_url",
    "thumbnail_url",
    "medium_url",
    "picture_url",
    "xl_picture_url",

    "host_url",
    "host_name",
    "host_about",
    "host_thumbnail_url",
    "host_picture_url",
    "host_location",
    "host_neighbourhood",
    "host_verifications",
    "host_has_profile_pic",
    "host_identity_verified",
    "host_acceptance_rate",

    "scrape_id",
    "last_scraped",
    "calendar_last_scraped",
    "calendar_updated",
    "license",
    "jurisdiction_names",

    "street",
    "zipcode",
    "market",
    "smart_location",
    "country_code",
    "country",

    "neighbourhood_group_cleansed",
    "square_feet",
    "weekly_price",
    "monthly_price",

    "maximum_nights_avg_ntm",
    "minimum_minimum_nights",
    "maximum_maximum_nights",
    "minimum_nights_avg_ntm",
    "maximum_minimum_nights",
    "minimum_maximum_nights",
    "number_of_reviews_ltm",

    "calculated_host_listings_count_shared_rooms",
    "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms",

    "experiences_offered",
    "bed_type",
    "has_availability",
    "is_business_travel_ready",
    "requires_license"
]

dataset = dataset.drop(columns=colunas_remover, errors="ignore")

In [25]:
dataset.shape

(902210, 49)

In [26]:
(dataset.isnull().sum()
 .sort_values(ascending=False)
 [dataset.isnull().sum() > 0])

review_scores_checkin          448650
review_scores_accuracy         448586
review_scores_location         448553
review_scores_value            448551
review_scores_communication    448415
review_scores_cleanliness      448413
review_scores_rating           448016
first_review                   424288
reviews_per_month              424288
last_review                    424249
security_deposit               421280
host_response_rate             401846
host_response_time             401843
cleaning_fee                   313506
neighbourhood                   58077
beds                             2502
bathrooms                        1724
city                             1604
state                            1288
bedrooms                          850
host_total_listings_count         460
host_since                        460
host_listings_count               460
host_is_superhost                 460
dtype: int64

In [27]:
#tratando valores nulos
dataset["beds"] = dataset["beds"].fillna(dataset["beds"].median())
dataset["bathrooms"] = dataset["bathrooms"].fillna(dataset["bathrooms"].median())
dataset["bedrooms"] = dataset["bedrooms"].fillna(dataset["bedrooms"].median())

dataset["first_review"] = dataset["first_review"].fillna("Não informado")
dataset["last_review"] = dataset["last_review"].fillna("Não informado")

dataset["neighbourhood"] = dataset["neighbourhood"].fillna("Não informado")
dataset["city"] = dataset["city"].fillna("Não informado")
dataset["state"] = dataset["state"].fillna("Não informado")
dataset["host_since"] = dataset["host_since"].fillna("Não informado")
dataset["host_response_time"] = dataset["host_response_time"].fillna("Não informado")

dataset["security_deposit"] = dataset["security_deposit"].fillna(0)
dataset["cleaning_fee"] = dataset["cleaning_fee"].fillna(0)

dataset["host_listings_count"] = dataset["host_listings_count"].fillna(0)
dataset["host_total_listings_count"] = dataset["host_total_listings_count"].fillna(0)

dataset["host_is_superhost"] = dataset["host_is_superhost"].fillna("f")

dataset["host_response_rate"] = dataset["host_response_rate"].astype(str)
dataset["host_response_rate"] = dataset["host_response_rate"].str.replace("%", "")
dataset["host_response_rate"] = dataset["host_response_rate"].astype(float)
dataset["host_response_rate"] = dataset["host_response_rate"].fillna(0)

colunas_review = [
"review_scores_rating",
"review_scores_accuracy",
"review_scores_cleanliness",
"review_scores_checkin",
"review_scores_communication",
"review_scores_location",
"review_scores_value",
"reviews_per_month"
]

for coluna in colunas_review:
    dataset[coluna] = dataset[coluna].fillna(0)

In [28]:
dataset.isnull().sum().sum()

np.int64(0)

In [29]:
#verificando tipos de dados em cada coluna
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 902210 entries, 0 to 902209
Data columns (total 49 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   id                                902210 non-null  int64  
 1   host_id                           902210 non-null  int64  
 2   host_since                        902210 non-null  str    
 3   host_response_time                902210 non-null  str    
 4   host_response_rate                902210 non-null  float64
 5   host_is_superhost                 902210 non-null  str    
 6   host_listings_count               902210 non-null  float64
 7   host_total_listings_count         902210 non-null  float64
 8   neighbourhood                     902210 non-null  str    
 9   neighbourhood_cleansed            902210 non-null  str    
 10  city                              902210 non-null  str    
 11  state                             902210 non-null  str    
 12 

In [30]:
colunas_dinheiro = [
    "price",
    "security_deposit",
    "cleaning_fee",
    "extra_people"
]

for coluna in colunas_dinheiro:
    dataset[coluna] = (
        dataset[coluna]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

dataset["host_since"] = pd.to_datetime(dataset["host_since"], errors="coerce")
dataset["first_review"] = pd.to_datetime(dataset["first_review"], errors="coerce")
dataset["last_review"] = pd.to_datetime(dataset["last_review"], errors="coerce")

colunas_boolean = [
    "host_is_superhost",
    "instant_bookable",
    "require_guest_profile_picture",
    "require_guest_phone_verification",
    "is_location_exact"
]

for coluna in colunas_boolean:
    dataset[coluna] = dataset[coluna].map({"f":0, "t":1})

In [31]:
dataset.nunique().sort_values()

host_is_superhost                        2
is_location_exact                        2
require_guest_phone_verification         2
require_guest_profile_picture            2
instant_bookable                         2
room_type                                4
host_response_time                       5
cancellation_policy                      6
review_scores_value                     10
review_scores_accuracy                  10
review_scores_checkin                   10
review_scores_cleanliness               10
review_scores_location                  10
review_scores_communication             10
bedrooms                                21
guests_included                         22
accommodates                            24
availability_30                         31
bathrooms                               38
property_type                           50
beds                                    51
review_scores_rating                    60
availability_60                         61
state      

In [33]:
dataset["price"].describe()

count    902210.000000
mean        647.605448
std        1704.745062
min           0.000000
25%         151.000000
50%         296.000000
75%         601.000000
max      138288.000000
Name: price, dtype: float64

In [34]:
#tratando outliers
dataset = dataset[dataset["price"] < 1000]

In [35]:
#criando colunas uteis
dataset["total_amenities"] = dataset["amenities"].str.count(",") + 1

In [36]:
#criando uma amostra
dataset_sample = dataset.sample(10000)

In [ ]:
#EDA

In [ ]:
#encoding

In [ ]:
#modelo de previsão

In [ ]:
#analise do melhor modelo

In [ ]:
#ajustes e melhorias